# Fashion-MNIST CNN + Optuna 베이지안 하이퍼파라미터 탐색

기존 `fashion_pytorch_cnn.ipynb`를 확장한 버전입니다.

핵심 변경 사항은 다음 세 가지입니다.

1. **Optuna(TPE 샘플러)를 이용한 베이지안 하이퍼파라미터 탐색**
2. **손실함수 / 최적화알고리즘 / 활성화함수를 레지스트리(딕셔너리)로 관리** — 문자열 이름만 바꾸면 교체 가능
3. **2단계 학습 전략** — 일부 데이터로 빠르게 탐색 → 최적 조합으로 전체 데이터 재학습

> **중요: 탐색 공간은 좁게, 기준점(baseline)은 등록하고 시작하세요.**
>
> trial 횟수가 10~20회 정도로 적은데 탐색 공간이 너무 넓으면
> 베이지안 탐색은 사실상 랜덤 탐색과 다를 게 없어서,
> 검증된 조합 하나로 오래 학습하는 것보다 결과가 오히려 나쁠 수 있습니다.
> 그래서 이 노트북은
> - 영향이 큰 하이퍼파라미터(학습률, dropout, 옵티마이저 등)만 좁은 범위로 탐색하고
> - 기존 노트북의 검증된 기본 조합을 첫 trial로 등록(`enqueue_trial`)하여
>   **탐색 결과가 최소한 기본 조합 이하로는 떨어지지 않게** 합니다.

기본 탐색 대상:

| 항목 | 탐색 범위 |
|---|---|
| 학습률 (lr) | 3e-4 ~ 5e-3 (로그 스케일) |
| 배치 크기 | 128, 256 |
| FC 은닉 노드 수 | 128, 256 |
| Dropout 비율 | 0.0 ~ 0.3 |
| 최적화 알고리즘 | adam, adamw |
| 활성화 함수 | relu, gelu |

손실함수, conv 채널 수, 커널 크기는 기본값으로 고정했지만,
레지스트리 구조 덕분에 `objective` 함수에 `trial.suggest_categorical` 한 줄만 추가하면
언제든 탐색 대상에 넣을 수 있습니다.

> **GPU(코랩 런타임 유형: GPU) 사용을 강력히 권장합니다.**
> CPU에서는 trial당 수십 초~수 분이 걸립니다. CPU라면 `N_TRIALS`를 8 정도로 줄이세요.

In [ ]:
# ===============================
# 0. Optuna 설치 확인
# ===============================

# Optuna는 베이지안 최적화 기반의 하이퍼파라미터 탐색 라이브러리입니다.
# 설치되어 있지 않으면 자동으로 설치합니다. (코랩에서는 보통 설치가 필요합니다)
try:
    import optuna
except ImportError:
    %pip install optuna -q
    import optuna

print("Optuna version:", optuna.__version__)

In [ ]:
# ===============================
# 1. 라이브러리 임포트 및 환경 설정
# ===============================

import numpy as np
import matplotlib.pyplot as plt
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, f1_score

# 재현성을 위해 난수 시드를 고정합니다.
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# GPU 사용 가능 여부에 따라 연산 장치를 선택합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

# Optuna의 로그가 너무 길게 출력되지 않도록 수준을 조정합니다. (원하면 INFO로 변경)
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 2. 레지스트리: 활성화 함수 / 최적화 알고리즘 / 손실 함수

핵심 아이디어는 **"이름(문자열) → 생성 함수" 딕셔너리**입니다.

- 모델, 옵티마이저, 손실함수를 만들 때 항상 이 레지스트리를 통해서 생성합니다.
- 따라서 `CONFIG["activation"] = "gelu"` 처럼 문자열만 바꾸면 즉시 교체됩니다.
- Optuna는 이 딕셔너리의 key 목록을 탐색 후보로 그대로 사용합니다.
- 새 항목을 추가하고 싶으면 딕셔너리에 한 줄만 추가하면 됩니다.

In [ ]:
# ===============================
# 2. 활성화/옵티마이저/손실 레지스트리
# ===============================

# --- 활성화 함수 레지스트리 ---
# value는 "호출하면 새 활성화 모듈을 만들어 주는 함수"입니다.
ACTIVATIONS = {
    "relu":       lambda: nn.ReLU(),
    "leaky_relu": lambda: nn.LeakyReLU(0.01),
    "elu":        lambda: nn.ELU(),
    "gelu":       lambda: nn.GELU(),
    "silu":       lambda: nn.SiLU(),     # Swish라고도 부릅니다.
    "tanh":       lambda: nn.Tanh(),
}

# --- 최적화 알고리즘 레지스트리 ---
# value는 (모델 파라미터, 학습률)을 받아 옵티마이저를 만들어 주는 함수입니다.
OPTIMIZERS = {
    "adam":    lambda params, lr: optim.Adam(params, lr=lr),
    "adamw":   lambda params, lr: optim.AdamW(params, lr=lr, weight_decay=1e-4),
    "rmsprop": lambda params, lr: optim.RMSprop(params, lr=lr),
    "sgd":     lambda params, lr: optim.SGD(params, lr=lr, momentum=0.9),
}


# NLLLoss는 로그 확률을 입력으로 받으므로,
# logits에 log_softmax를 먼저 적용해 주는 래퍼 클래스를 만듭니다.
# 수학적으로는 CrossEntropyLoss와 동일하지만, 손실함수를
# "갈아끼우는" 구조를 보여주기 위해 포함했습니다.
class NLLWithLogSoftmax(nn.Module):
    def __init__(self):
        super().__init__()
        self.nll = nn.NLLLoss()

    def forward(self, logits, target):
        return self.nll(F.log_softmax(logits, dim=1), target)


# --- 손실 함수 레지스트리 (다중 분류용) ---
LOSSES = {
    "cross_entropy":  lambda: nn.CrossEntropyLoss(),
    # label smoothing: 정답을 1.0이 아니라 0.9~0.95 정도로 부드럽게 만들어
    # 과적합과 과신(over-confidence)을 줄이는 기법입니다.
    "ce_smooth_0.05": lambda: nn.CrossEntropyLoss(label_smoothing=0.05),
    "ce_smooth_0.1":  lambda: nn.CrossEntropyLoss(label_smoothing=0.1),
    "nll_logsoftmax": lambda: NLLWithLogSoftmax(),
}


# 레지스트리에서 객체를 꺼내는 헬퍼 함수들입니다.
def make_activation(name):
    return ACTIVATIONS[name]()

def make_optimizer(name, params, lr):
    return OPTIMIZERS[name](params, lr)

def make_loss(name):
    return LOSSES[name]()

print("활성화 함수 후보:", list(ACTIVATIONS.keys()))
print("옵티마이저 후보:", list(OPTIMIZERS.keys()))
print("손실 함수 후보:", list(LOSSES.keys()))

## 3. 기본 설정(CONFIG)

수동으로 실험할 때는 이 딕셔너리의 값만 바꾸면 됩니다.

예시:
```python
CONFIG["activation"] = "gelu"
CONFIG["optimizer"] = "adamw"
CONFIG["loss"] = "ce_smooth_0.1"
```

In [ ]:
# ===============================
# 3. 기본 설정 (수동 실험용 기준값)
# ===============================

CONFIG = {
    # --- 교체 가능한 구성 요소 (레지스트리의 key 문자열) ---
    "loss":       "cross_entropy",
    "optimizer":  "adam",
    "activation": "relu",

    # --- 숫자형 하이퍼파라미터 ---
    "lr":             1e-3,
    "batch_size":     300,
    "conv1_channels": 32,
    "conv2_channels": 64,
    "fc_units":       128,
    "dropout":        0.0,
    "kernel_size":    2,
    "epochs":         20,
}

# 클래스 수와 클래스 이름 (Fashion-MNIST 고정값)
M_CLASS = 10
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

In [ ]:
# ===============================
# 4. 데이터 준비
# ===============================

# ToTensor: 이미지를 (1, 28, 28) 텐서로 바꾸고 픽셀값을 0~1로 정규화합니다.
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.FashionMNIST(root="./data", train=True,  download=True, transform=transform)
test_dataset  = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

print("학습용 데이터 개수:", len(train_dataset))
print("평가용 데이터 개수:", len(test_dataset))

# --- 탐색용 서브셋 만들기 ---
# 하이퍼파라미터 탐색은 여러 번 학습을 반복하므로,
# 전체 60,000장을 다 쓰면 너무 오래 걸립니다.
# 학습 데이터에서 일부만 떼어 "탐색용 학습/검증 세트"를 만듭니다.
SEARCH_TRAIN_SIZE = 10000   # 탐색 단계에서 학습에 사용할 데이터 수
SEARCH_VALID_SIZE = 2000    # 탐색 단계에서 검증에 사용할 데이터 수

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(train_dataset), generator=g)

search_train = Subset(train_dataset, perm[:SEARCH_TRAIN_SIZE].tolist())
search_valid = Subset(train_dataset, perm[SEARCH_TRAIN_SIZE:SEARCH_TRAIN_SIZE + SEARCH_VALID_SIZE].tolist())

print("탐색용 학습 데이터:", len(search_train))
print("탐색용 검증 데이터:", len(search_valid))

## 5. 설정 기반 CNN 모델

기존 모델과 구조는 같지만, 채널 수 / FC 노드 수 / 커널 크기 / Dropout / 활성화 함수를
모두 생성자 인자로 받도록 일반화했습니다.

`28x28 → (pool) 14x14 → (pool) 7x7` 구조는 유지되므로
FC 입력 크기는 `conv2_channels * 7 * 7`로 자동 계산됩니다.

In [ ]:
# ===============================
# 5. 설정 기반 CNN 모델 정의
# ===============================

class FashionCNN(nn.Module):
    def __init__(self, conv1_channels=32, conv2_channels=64, fc_units=128,
                 dropout=0.0, kernel_size=2, activation="relu", n_classes=10):
        super().__init__()

        # 활성화 함수는 레지스트리에서 이름으로 생성합니다.
        self.act = make_activation(activation)

        # 합성곱층 2개: padding="same"으로 가로세로 크기를 유지합니다.
        self.conv1 = nn.Conv2d(1, conv1_channels, kernel_size=kernel_size, padding="same")
        self.conv2 = nn.Conv2d(conv1_channels, conv2_channels, kernel_size=kernel_size, padding="same")

        # 풀링층: 2x2 최대 풀링으로 크기를 절반으로 줄입니다.
        self.pool = nn.MaxPool2d(kernel_size=2)

        # Dropout: 학습 시 일부 노드를 무작위로 꺼서 과적합을 줄입니다.
        self.dropout = nn.Dropout(dropout)

        # 완전연결층: 28 → 14 → 7 이므로 입력 크기는 conv2_channels * 7 * 7 입니다.
        self.fc1 = nn.Linear(conv2_channels * 7 * 7, fc_units)
        self.fc2 = nn.Linear(fc_units, n_classes)

    def forward(self, x):
        x = self.pool(self.act(self.conv1(x)))   # (B, c1, 14, 14)
        x = self.pool(self.act(self.conv2(x)))   # (B, c2, 7, 7)
        x = x.view(x.size(0), -1)                # (B, c2*7*7)
        x = self.dropout(self.act(self.fc1(x)))  # (B, fc_units)
        x = self.fc2(x)                          # (B, 10) — logits
        return x


def build_from_config(config):
    # CONFIG 딕셔너리 하나로 모델/손실함수/옵티마이저를 모두 생성하는 헬퍼입니다.
    # 구성 요소를 바꾸고 싶으면 config의 문자열만 바꾸면 됩니다.
    model = FashionCNN(
        conv1_channels=config["conv1_channels"],
        conv2_channels=config["conv2_channels"],
        fc_units=config["fc_units"],
        dropout=config["dropout"],
        kernel_size=config["kernel_size"],
        activation=config["activation"],
        n_classes=M_CLASS,
    ).to(device)

    criterion = make_loss(config["loss"])
    optimizer = make_optimizer(config["optimizer"], model.parameters(), config["lr"])
    return model, criterion, optimizer


# 동작 확인: 기본 CONFIG로 모델을 만들어 더미 입력을 통과시켜 봅니다.
_model, _criterion, _optimizer = build_from_config(CONFIG)
print(_model)
print("더미 출력 모양:", _model(torch.randn(1, 1, 28, 28).to(device)).shape)

In [ ]:
# ===============================
# 6. 학습/평가 함수 정의
# ===============================

def train_one_epoch(model, loader, criterion, optimizer, device):
    # 한 epoch 동안 모델을 학습시키고 (평균 손실, 정확도)를 반환합니다.
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()           # 이전 배치의 기울기 초기화
        outputs = model(images)         # 순전파: logits 계산
        loss = criterion(outputs, labels)
        loss.backward()                 # 역전파: 기울기 계산
        optimizer.step()                # 가중치 업데이트

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    # 평가 데이터에 대해 (평균 손실, 정확도, 정답 배열, 예측 배열)을 반환합니다.
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return running_loss / total, correct / total, np.array(all_labels), np.array(all_preds)

## 7. Optuna 목적 함수(objective)

Optuna는 `objective(trial)` 함수를 반복 호출하면서 탐색합니다.

- `trial.suggest_*()` : 탐색 범위에서 다음에 시도할 값을 뽑습니다. TPE 샘플러는 지금까지의 결과를 바탕으로 좋은 영역을 더 자주 뽑습니다.
- `trial.report()` + `trial.should_prune()` : 성능이 나쁜 trial을 조기 중단(pruning)하여 시간을 절약합니다.
- 반환값: **검증 정확도** (이 값을 최대화하는 방향으로 탐색합니다)

**탐색 공간 설계 원칙** — trial 수가 적을 때는 다음을 지켜야 단일 조합 학습보다 좋은 결과가 나옵니다.

1. **영향이 큰 것만 탐색**: 보통 학습률 > dropout/정규화 > 옵티마이저 > 모델 크기 순으로 영향이 큽니다.
2. **범위를 좁게**: 학습률을 1e-4~1e-2처럼 넓게 잡으면 대부분의 trial이 낭비됩니다.
   이미 잘 되는 값(1e-3) 주변으로 좁히는 것이 효율적입니다.
3. **카테고리 후보 줄이기**: 손실함수 4종 × 옵티마이저 4종 × 활성화 6종이면
   조합만 96가지라 20 trial로는 탐색 자체가 불가능합니다.

In [ ]:
# ===============================
# 7. Optuna 목적 함수 정의
# ===============================

SEARCH_EPOCHS = 3   # 탐색 단계에서는 epoch을 적게 사용합니다.

def objective(trial):
    # 기본 설정(CONFIG)을 복사한 뒤, 탐색할 항목만 trial 값으로 덮어씁니다.
    # 손실함수 / conv 채널 / 커널 크기는 CONFIG 기본값으로 고정합니다.
    # 탐색에 추가하고 싶으면 아래에 suggest_* 한 줄만 추가하면 됩니다.
    # 예: "loss": trial.suggest_categorical("loss", list(LOSSES.keys())),
    config = dict(CONFIG)
    config.update({
        "optimizer":  trial.suggest_categorical("optimizer", ["adam", "adamw"]),
        "activation": trial.suggest_categorical("activation", ["relu", "gelu"]),
        "lr":         trial.suggest_float("lr", 3e-4, 5e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [128, 256]),
        "fc_units":   trial.suggest_categorical("fc_units", [128, 256]),
        "dropout":    trial.suggest_float("dropout", 0.0, 0.3),
    })

    # 탐색용 서브셋으로 DataLoader를 만듭니다.
    train_loader = DataLoader(search_train, batch_size=config["batch_size"], shuffle=True)
    valid_loader = DataLoader(search_valid, batch_size=512, shuffle=False)

    # config 하나로 모델/손실/옵티마이저를 일괄 생성합니다.
    model, criterion, optimizer = build_from_config(config)

    best_acc = 0.0
    for epoch in range(1, SEARCH_EPOCHS + 1):
        train_one_epoch(model, train_loader, criterion, optimizer, device)
        _, val_acc, _, _ = evaluate(model, valid_loader, criterion, device)
        best_acc = max(best_acc, val_acc)

        # 중간 결과를 보고하고, 성능이 나쁘면 조기 중단합니다.
        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_acc

In [ ]:
# ===============================
# 8. 베이지안 탐색 실행
# ===============================

# 시도 횟수입니다. GPU에서는 20~30까지 늘려도 좋고, CPU에서는 8 정도로 줄이세요.
N_TRIALS = 15

study = optuna.create_study(
    direction="maximize",                       # 검증 정확도 최대화
    # n_startup_trials=5: 처음 5번만 무작위 탐색 후 베이지안 모델링을 시작합니다.
    # (기본값 10은 전체 trial 수가 15일 때 너무 큽니다)
    sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=5),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
)

# ★ 기존 노트북에서 검증된 기본 조합을 첫 trial로 등록합니다.
# 탐색이 이 조합을 반드시 한 번 평가하므로,
# "탐색으로 찾은 최적값"이 기본 조합보다 나빠질 수 없습니다.
study.enqueue_trial({
    "optimizer":  "adam",
    "activation": "relu",
    "lr":         1e-3,
    "batch_size": 256,
    "fc_units":   128,
    "dropout":    0.0,
})

begin = time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print("탐색 시간: {:.1f}초".format(time() - begin))

print()
print("=== 최적 결과 ===")
print("최고 검증 정확도: {:.2f}%".format(study.best_value * 100))
print("최적 하이퍼파라미터:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# ===============================
# 9. 탐색 결과 분석
# ===============================

# 전체 trial 결과를 표로 확인합니다. (상위 10개)
df = study.trials_dataframe(attrs=("number", "value", "params", "state"))
display(df.sort_values("value", ascending=False).head(10))

# 최적화 진행 과정과 하이퍼파라미터 중요도를 시각화합니다.
try:
    from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

    plot_optimization_history(study)
    plt.tight_layout()
    plt.show()

    plot_param_importances(study)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("시각화 생략:", e)

## 10. 최적 하이퍼파라미터로 전체 데이터 재학습

탐색 단계에서는 데이터 일부 + 3 epoch만 사용했으므로,
이제 찾은 최적 조합으로 **전체 학습 데이터(60,000장)** 를 다시 학습하고
테스트 데이터(10,000장)로 최종 성능을 평가합니다.

In [ ]:
# ===============================
# 10. 최종 학습 (전체 데이터)
# ===============================

FINAL_EPOCHS = 20   # 원본 노트북과 동일한 20 epoch — 공평한 비교를 위해 맞췄습니다.
                    # (시간이 부족하면 줄이되, 원본과 비교할 때는 같은 값을 쓰세요)

# 기본 CONFIG에 탐색으로 찾은 최적값을 덮어씁니다.
best_config = dict(CONFIG)
best_config.update(study.best_params)
best_config["epochs"] = FINAL_EPOCHS
print("최종 학습 설정:", best_config)

train_loader = DataLoader(train_dataset, batch_size=best_config["batch_size"], shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=512, shuffle=False)

model, criterion, optimizer = build_from_config(best_config)

begin = time()
train_losses, train_accuracies = [], []

for epoch in range(1, best_config["epochs"] + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    print(f"Epoch [{epoch:02d}/{best_config['epochs']}] "
          f"Loss: {train_loss:.4f} Accuracy: {train_acc * 100:.2f}%")

print("총 학습 시간: {:.1f}초".format(time() - begin))

In [ ]:
# ===============================
# 11. 학습 과정 시각화
# ===============================

epochs = range(1, best_config["epochs"] + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_losses, marker="o", label="Train Loss")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs, train_accuracies, marker="o", label="Train Accuracy")
axes[1].set_title("Training Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ===============================
# 12. 테스트 데이터 최종 평가
# ===============================

test_loss, test_acc, truth, pred = evaluate(model, test_loader, criterion, device)

print("최종 테스트 손실값: {:.4f}".format(test_loss))
print("최종 정확도: {:.2f}%".format(test_acc * 100))

# F1 점수
f1 = f1_score(truth, pred, average="micro")
print("F1 점수: {:.3f}".format(f1))

# 혼동 행렬 시각화
cm = confusion_matrix(truth, pred)

plt.figure(figsize=(8, 8))
plt.imshow(cm)
plt.title("Confusion Matrix (Best Config)")
plt.colorbar()
plt.xticks(np.arange(M_CLASS), class_names, rotation=45, ha="right")
plt.yticks(np.arange(M_CLASS), class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
for i in range(M_CLASS):
    for j in range(M_CLASS):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout()
plt.show()

## 13. 수동 실험 방법 (Optuna 없이 구성 요소만 교체)

Optuna 탐색 없이 직접 조합을 바꿔 실험하고 싶을 때는
아래처럼 `CONFIG` 값만 수정한 뒤 `build_from_config`로 다시 만들면 됩니다.

```python
my_config = dict(CONFIG)
my_config["loss"] = "ce_smooth_0.1"      # 손실 함수 교체
my_config["optimizer"] = "adamw"         # 옵티마이저 교체
my_config["activation"] = "gelu"         # 활성화 함수 교체
my_config["lr"] = 5e-4

model, criterion, optimizer = build_from_config(my_config)
# 이후 train_one_epoch / evaluate를 그대로 사용하면 됩니다.
```

새로운 구성 요소를 추가하고 싶다면 레지스트리에 한 줄만 추가하세요.

```python
ACTIVATIONS["mish"] = lambda: nn.Mish()
OPTIMIZERS["adagrad"] = lambda params, lr: optim.Adagrad(params, lr=lr)
LOSSES["ce_smooth_0.2"] = lambda: nn.CrossEntropyLoss(label_smoothing=0.2)
```